In [ ]:
%pip install -U google-genai python-dotenv ipywidgets

In [ ]:
from getpass import getpass
from pathlib import Path

api_key = getpass("Enter your new Gemini API key: ")

if not api_key.strip():
    raise ValueError("API key cannot be empty.")

Path(".env").write_text(
    f"GEMINI_API_KEY={api_key.strip()}\n",
    encoding="utf-8"
)

print("API key saved successfully!")

In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv(override=True)

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("API key not found.")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Say hello in one sentence."
)

print(response.text)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

title = widgets.HTML("""
<h1>✉️ AI Email & Message Rewriter</h1>
<p>Rewrite your messages with Gemini AI.</p>
""")

draft = widgets.Textarea(
    placeholder="Paste your email or message here...",
    description="Draft:",
    layout=widgets.Layout(width="100%", height="160px"),
    style={"description_width": "initial"}
)

tone = widgets.Dropdown(
    options=["Polite", "Direct", "Friendly"],
    value="Polite",
    description="Tone:",
    style={"description_width": "initial"}
)

button = widgets.Button(
    description="✨ Rewrite with AI",
    button_style="primary",
    icon="magic"
)

output = widgets.Output(
    layout=widgets.Layout(width="100%")
)

display(widgets.VBox([
    title,
    draft,
    tone,
    button,
    output
], layout=widgets.Layout(width="100%", gap="12px")))

In [ ]:
def rewrite_message(_):
    button.disabled = True

    with output:
        clear_output(wait=True)

        text = draft.value.strip()
        selected_tone = tone.value

        if not text:
            print("Please enter a message first.")
            button.disabled = False
            return

        try:
            print("Rewriting your message...")

            response = client.interactions.create(
                model="gemini-3.1-flash-lite",
                input=(
                    f"Rewrite the following message in a "
                    f"{selected_tone.lower()} tone. "
                    f"Keep the original meaning. "
                    f"Return only the rewritten message.\n\n"
                    f"{text}"
                )
            )

            clear_output(wait=True)
            print(response.output_text)

        except Exception as e:
            clear_output(wait=True)
            print(f"Something went wrong: {e}")

        finally:
            button.disabled = False


button.on_click(rewrite_message)

print("Gemini is connected to your Rewrite button!")

In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv(override=True)

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "API key not found. Check your .env file."
    )

client = genai.Client(api_key=api_key)

print("Gemini client is ready!")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Javascript
import json

title = widgets.HTML("""
<h2>✉️ AI Email & Message Rewriter</h2>
<p>Rewrite your message and copy the result.</p>
""")

draft = widgets.Textarea(
    placeholder="Type your original message here...",
    description="Draft:",
    layout=widgets.Layout(width="100%", height="120px")
)

tone = widgets.Dropdown(
    options=["Polite", "Direct", "Friendly"],
    value="Polite",
    description="Tone:"
)

rewrite_button = widgets.Button(
    description="✨ Rewrite with AI",
    button_style="primary"
)

result_box = widgets.Textarea(
    placeholder="Your rewritten message will appear here...",
    description="Result:",
    layout=widgets.Layout(width="100%", height="150px")
)

copy_button = widgets.Button(
    description="📋 Copy Rewritten Message",
    button_style="success"
)

status = widgets.HTML()

def rewrite_message(_):
    text = draft.value.strip()

    if not text:
        status.value = "Please enter a message in Draft."
        return

    rewrite_button.disabled = True
    status.value = "Rewriting..."

    try:
        response = client.interactions.create(
            model="gemini-3.1-flash-lite",
            input=(
                f"Rewrite this message in a {tone.value.lower()} tone. "
                "Keep the original meaning. Return only the rewritten message.\n\n"
                + text
            )
        )

        result_box.value = (response.output_text or "").strip()
        status.value = "Rewriting complete."

    except Exception as e:
        status.value = "Gemini error: " + str(e)

    finally:
        rewrite_button.disabled = False

def copy_message(_):
    text = result_box.value.strip()

    if not text:
        status.value = "Please rewrite a message first."
        return

    # Try to copy through the browser clipboard.
    js = """
    (async () => {
        try {
            await navigator.clipboard.writeText(""" + json.dumps(text) + """);
            alert("Copied! Try pasting into Notepad or WhatsApp.");
        } catch (err) {
            alert("Automatic copy is blocked. Use the manual copy steps below.");
        }
    })();
    """
    display(Javascript(js))
    status.value = (
        "If automatic copying fails: click inside Result, press Ctrl+A, "
        "then Ctrl+C. Click Notepad or WhatsApp and press Ctrl+V."
    )

rewrite_button.on_click(rewrite_message)
copy_button.on_click(copy_message)

display(widgets.VBox([
    title,
    draft,
    tone,
    rewrite_button,
    result_box,
    copy_button,
    status
], layout=widgets.Layout(width="100%", gap="10px")))